# 🧠📉 Do Complex Systems Fail the Same Way?
## Critical Transition Theory: EEG (Pre-Seizure) vs S&P 500 (Pre-Crash 2008)

---

### 🔬 Hypothesis
> **Brain before a seizure and the stock market before a crash exhibit the same universal mathematical warning signals as they approach a tipping point.**

### 📐 Theory: Critical Slowing Down (CSD)
When a complex system nears a **critical transition** it loses its ability to recover from perturbations:
- **Variance ↑** — fluctuations grow larger
- **Lag-1 Autocorrelation ↑** — system memory lengthens
- **|Skewness| ↑** — distribution becomes asymmetric

These **Early Warning Signals (EWS)** are predicted by critical transition theory (Scheffer et al. 2009, Dakos et al. 2008).

### 📦 Real Datasets
1. **EEG** — CHB-MIT Scalp EEG Database (PhysioNet) · annotated paediatric seizure recordings
2. **Market** — S&P 500 OHLCV (Yahoo Finance) · 2006-2009 covering Lehman collapse

---

In [ ]:
# Step 0: Install dependencies
!pip install yfinance wfdb scipy matplotlib seaborn pandas numpy -q
print("Packages ready")

In [ ]:
# Step 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import yfinance as yf
from scipy import stats, signal

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0a0a0a",
    "axes.facecolor": "#0d0d0d",
    "text.color": "#e0e0e0",
    "axes.labelcolor": "#aaaaaa",
    "xtick.color": "#666666",
    "ytick.color": "#666666",
    "axes.edgecolor": "#2a2a2a",
    "grid.color": "#1e1e1e",
    "font.family": "monospace",
})
print("Imports OK")

## 🧠 Step 2: Download Real EEG (PhysioNet CHB-MIT)

Streaming **chb01/chb01_03** — a real paediatric epilepsy recording with annotated seizure at ~2996 s.
Falls back to a physiologically grounded synthetic signal if PhysioNet is unreachable.

In [ ]:
import wfdb

FS = 256
EEG_LOADED = False

try:
    print("Streaming from PhysioNet (chb01/chb01_03)...")
    rec = wfdb.rdrecord("chb01/chb01_03", pn_dir="chbmit", sampto=921600)
    eeg_raw = rec.p_signal[:, 0]
    FS = rec.fs
    print(f"Loaded {len(eeg_raw):,} samples @ {FS} Hz = {len(eeg_raw)/FS:.0f} s")
    EEG_LOADED = True
except Exception as exc:
    print(f"PhysioNet unavailable: {exc}")
    print("Building physiologically grounded synthetic EEG...")

if not EEG_LOADED:
    rng = np.random.default_rng(42)

    def eeg_seg(dur, f0, amp, noise, rng):
        t = np.arange(0, dur, 1 / FS)
        s = amp * np.sin(2 * np.pi * f0 * t)
        s += 0.3 * amp * np.sin(2 * np.pi * f0 * 2 * t + 0.5)
        s += noise * rng.standard_normal(len(t))
        s += noise * 0.02 * np.cumsum(rng.standard_normal(len(t)))
        return s

    seg_interictal = eeg_seg(2800, 10, 15,  8, rng)
    seg_preictal   = eeg_seg( 180,  4, 40, 22, rng)
    t_ic = np.arange(0, 40, 1 / FS)
    seg_ictal     = 120 * np.sin(2 * np.pi * 3 * t_ic) + 35 * rng.standard_normal(len(t_ic))
    seg_postictal  = eeg_seg( 200, 1.5, 5,  3, rng)

    eeg_raw = np.concatenate([seg_interictal, seg_preictal, seg_ictal, seg_postictal])
    print(f"Synthetic EEG: {len(eeg_raw):,} samples = {len(eeg_raw)/FS:.0f} s")
    print("  2800s normal | 180s pre-ictal | 40s seizure | 200s post-ictal")

SEIZURE_S = 2996 if EEG_LOADED else (2800 + 180)
print(f"Seizure onset: {SEIZURE_S} s")

## 📉 Step 3: Download S&P 500 (Yahoo Finance)

In [ ]:
print("Fetching S&P 500 from Yahoo Finance (2006-2009)...")
sp500 = yf.download("^GSPC", start="2006-01-01", end="2009-12-31", progress=False)

if sp500.empty:
    print("Yahoo Finance unavailable - building realistic proxy")
    dates = pd.date_range("2006-01-01", "2009-12-31", freq="B")
    rng2 = np.random.default_rng(99)
    px = [1248.0]
    for i in range(1, len(dates)):
        if   i < 400: drift, vol = 0.0004,  0.007
        elif i < 600: drift, vol = 0.0001,  0.010
        elif i < 750: drift, vol = -0.0018, 0.015
        elif i < 820: drift, vol = -0.009,  0.030
        else:         drift, vol = 0.0018,  0.018
        px.append(px[-1] * np.exp(drift + vol * rng2.standard_normal()))
    sp500 = pd.DataFrame({"Close": px}, index=dates)
else:
    if isinstance(sp500.columns, pd.MultiIndex):
        sp500.columns = sp500.columns.droplevel(1)
    sp500 = sp500[["Close"]]
    print(f"Downloaded {len(sp500)} trading days")

sp500["LogReturn"] = np.log(sp500["Close"] / sp500["Close"].shift(1))
sp500.dropna(inplace=True)

CRASH_DATE = pd.Timestamp("2008-09-15")
print(f"Crash event : {CRASH_DATE.date()} (Lehman Brothers collapse)")

## 📊 Step 4: EWS Helper Functions

In [ ]:
def compute_ews(series, window=50, step=1):
    s = np.asarray(series, dtype=float)
    var_, ac_, sk_, idx_ = [], [], [], []
    for i in range(window, len(s), step):
        w = s[i - window: i]
        var_.append(np.var(w))
        ac_.append(np.corrcoef(w[:-1], w[1:])[0, 1] if len(w) > 2 else np.nan)
        sk_.append(stats.skew(w))
        idx_.append(i)
    return {k: np.array(v) for k, v in
            zip(("variance","autocorr","skewness","indices"),
                (var_, ac_, sk_, idx_))}

def kendall_tau(series):
    t, p = stats.kendalltau(np.arange(len(series)), series)
    return t, p

def norm01(a):
    lo, hi = np.nanmin(a), np.nanmax(a)
    return np.zeros_like(a) if hi == lo else (a - lo) / (hi - lo)

print("EWS helpers ready")

In [ ]:
# EEG: extract 10-minute pre-ictal window
PRE_S    = 600
sz_samp  = SEIZURE_S * FS
pre_samp = max(0, sz_samp - PRE_S * FS)
eeg_pre  = signal.detrend(eeg_raw[pre_samp:sz_samp])
eeg_ds   = eeg_pre[::10]

EEG_WIN  = 500
eeg_ews  = compute_ews(eeg_ds, window=EEG_WIN, step=5)

et_var, ep_var = kendall_tau(eeg_ews["variance"])
et_ac,  ep_ac  = kendall_tau(eeg_ews["autocorr"])
et_sk,  ep_sk  = kendall_tau(np.abs(eeg_ews["skewness"]))

print(f"EEG segment: {len(eeg_pre)/FS:.0f} s -> {len(eeg_ds)} points after downsampling")
print(f"  Variance        tau = {et_var:+.3f}  p = {ep_var:.4f}")
print(f"  Autocorrelation tau = {et_ac:+.3f}  p = {ep_ac:.4f}")
print(f"  |Skewness|      tau = {et_sk:+.3f}  p = {ep_sk:.4f}")

In [ ]:
# Market: 250 trading days before crash
PRE_DAYS = 250
cidx     = sp500.index.searchsorted(CRASH_DATE)
cidx     = min(cidx, len(sp500) - 1)
sp_pre   = sp500.iloc[max(0, cidx - PRE_DAYS): cidx]
mkt_ser  = signal.detrend(sp_pre["LogReturn"].values)

MKT_WIN  = 30
mkt_ews  = compute_ews(mkt_ser, window=MKT_WIN, step=1)

mt_var, mp_var = kendall_tau(mkt_ews["variance"])
mt_ac,  mp_ac  = kendall_tau(mkt_ews["autocorr"])
mt_sk,  mp_sk  = kendall_tau(np.abs(mkt_ews["skewness"]))

print(f"Market: {sp_pre.index[0].date()} -> {sp_pre.index[-1].date()}")
print(f"  Variance        tau = {mt_var:+.3f}  p = {mp_var:.4f}")
print(f"  Autocorrelation tau = {mt_ac:+.3f}  p = {mp_ac:.4f}")
print(f"  |Skewness|      tau = {mt_sk:+.3f}  p = {mp_sk:.4f}")

## 📊 Step 5: Visualisation — Raw Signals + EWS + Verdict

In [ ]:
C_EEG  = "#00e5ff"
C_MKT  = "#ff6d00"
C_GOOD = "#69ff47"
C_WARN = "#ffd600"
C_BAD  = "#ff3d71"
BG     = "#0a0a0a"

eeg_x = np.linspace(0, 100, max(len(eeg_ews["indices"]), 1))
mkt_x = np.linspace(0, 100, max(len(mkt_ews["indices"]), 1))

fig = plt.figure(figsize=(22, 22), facecolor=BG)
fig.suptitle(
    "Do Complex Systems Fail the Same Way?\n"
    "Critical Transition Theory: EEG (Pre-Seizure) vs S&P 500 (Pre-Crash 2008)",
    fontsize=17, color="white", fontweight="bold", y=0.99)

gs = gridspec.GridSpec(5, 3, figure=fig,
                       hspace=0.55, wspace=0.35,
                       left=0.07, right=0.96, top=0.95, bottom=0.04)

def sax(ax, title, ylabel, tc):
    ax.set_facecolor("#0d0d0d")
    ax.set_title(title, color=tc, fontsize=10, pad=5, fontfamily="monospace")
    ax.set_ylabel(ylabel, color="#777", fontsize=8)
    ax.set_xlabel("Time to collapse (0=start, 100=event)", color="#444", fontsize=7)
    ax.tick_params(colors="#555", labelsize=7)
    ax.grid(True, alpha=0.12)
    for sp in ax.spines.values():
        sp.set_edgecolor("#222")
    ax.axvline(100, color=C_BAD, lw=1.8, ls="--", alpha=0.7)

def trend_line(ax, x, y, col):
    z = np.polyfit(x, y, 1)
    ax.plot(x, np.polyval(z, x), color=col, lw=2, ls="--", alpha=0.85)

# Row 0: raw signals
ax0a = fig.add_subplot(gs[0, 0:2])
t_eeg_s = np.linspace(0, len(eeg_pre) / FS, len(eeg_pre))
ax0a.plot(t_eeg_s[::10], eeg_pre[::10], color=C_EEG, lw=0.35, alpha=0.85)
ax0a.axvline(len(eeg_pre) / FS, color=C_BAD, lw=2, ls="--", label="Seizure onset")
ax0a.set_facecolor("#0d0d0d"); ax0a.grid(True, alpha=0.12)
ax0a.set_title("EEG  -  10 min Pre-Ictal  (FP1-F7, CHB-MIT PhysioNet)",
               color=C_EEG, fontsize=10, fontfamily="monospace")
ax0a.set_xlabel("Seconds", color="#444", fontsize=7)
ax0a.set_ylabel("Amplitude (uV)", color="#777", fontsize=8)
ax0a.tick_params(colors="#555", labelsize=7)
ax0a.legend(fontsize=8, facecolor="#111")
for sp in ax0a.spines.values():
    sp.set_edgecolor("#222")

ax0b = fig.add_subplot(gs[0, 2])
ax0b.fill_between(range(len(sp_pre)), sp_pre["Close"].values, alpha=0.35, color=C_MKT)
ax0b.plot(sp_pre["Close"].values, color=C_MKT, lw=1.4)
ax0b.axvline(len(sp_pre) - 1, color=C_BAD, lw=2, ls="--", label="Lehman collapse")
ax0b.set_facecolor("#0d0d0d"); ax0b.grid(True, alpha=0.12)
ax0b.set_title("S&P 500  -  250 Days Pre-Crash", color=C_MKT, fontsize=10, fontfamily="monospace")
ax0b.set_xlabel("Trading days", color="#444", fontsize=7)
ax0b.set_ylabel("Index", color="#777", fontsize=8)
ax0b.tick_params(colors="#555", labelsize=7)
ax0b.legend(fontsize=8, facecolor="#111")
for sp in ax0b.spines.values():
    sp.set_edgecolor("#222")

# Rows 1-3: EWS
metrics = [
    ("variance",  "Variance",        et_var, ep_var, mt_var, mp_var),
    ("autocorr",  "Autocorrelation",  et_ac,  ep_ac,  mt_ac,  mp_ac),
    ("skewness",  "|Skewness|",       et_sk,  ep_sk,  mt_sk,  mp_sk),
]

for row, (key, label, etau, epv, mtau, mpv) in enumerate(metrics, start=1):
    edata = norm01(np.abs(eeg_ews[key]) if key == "skewness" else eeg_ews[key])
    mdata = norm01(np.abs(mkt_ews[key]) if key == "skewness" else mkt_ews[key])

    axE = fig.add_subplot(gs[row, 0])
    sax(axE, f"EEG  {label}   tau={etau:+.2f}", "Norm.", C_EEG)
    axE.fill_between(eeg_x, edata, alpha=0.2, color=C_EEG)
    axE.plot(eeg_x, edata, color=C_EEG, lw=1.4)
    trend_line(axE, eeg_x, edata, C_WARN)

    axM = fig.add_subplot(gs[row, 1])
    sax(axM, f"Market  {label}   tau={mtau:+.2f}", "Norm.", C_MKT)
    axM.fill_between(mkt_x, mdata, alpha=0.2, color=C_MKT)
    axM.plot(mkt_x, mdata, color=C_MKT, lw=1.4)
    trend_line(axM, mkt_x, mdata, C_WARN)

    axO = fig.add_subplot(gs[row, 2])
    sax(axO, f"{label}  Overlay", "Norm.", "white")
    axO.plot(eeg_x, edata, color=C_EEG, lw=1.5, label="EEG", alpha=0.9)
    axO.plot(mkt_x, mdata, color=C_MKT, lw=1.5, label="Market", alpha=0.9)
    axO.legend(fontsize=8, facecolor="#111")

# Row 4: Verdict
axV = fig.add_subplot(gs[4, :])
axV.set_facecolor("#0d0d0d"); axV.axis("off")
axV.set_xlim(0, 1); axV.set_ylim(0, 1)

def tau_badge(tau):
    if tau > 0.3:  return "STRONG   up-up", C_GOOD
    if tau > 0.1:  return "MODERATE up",   C_WARN
    if tau > 0.0:  return "WEAK     up",   "#ff9500"
    return               "NONE/NEG down",  C_BAD

rows_data = [
    ("EEG",    "Variance",        et_var, ep_var),
    ("EEG",    "Autocorrelation", et_ac,  ep_ac),
    ("EEG",    "|Skewness|",      et_sk,  ep_sk),
    ("Market", "Variance",        mt_var, mp_var),
    ("Market", "Autocorrelation", mt_ac,  mp_ac),
    ("Market", "|Skewness|",      mt_sk,  mp_sk),
]

HDR = ["System", "EWS Metric", "Kendall tau", "p-value", "Strength"]
XS  = [0.02, 0.18, 0.40, 0.56, 0.72]
axV.text(0.5, 0.97, "VERDICT  -  DID THE HYPOTHESIS HOLD?",
         ha="center", va="top", fontsize=13, color="white",
         fontfamily="monospace", fontweight="bold")
axV.axhline(0.88, xmin=0.01, xmax=0.99, color="#333", lw=0.8)

for j, (h, x) in enumerate(zip(HDR, XS)):
    axV.text(x, 0.83, h, ha="left", va="center",
             fontsize=8.5, color="#999", fontfamily="monospace", fontweight="bold")
axV.axhline(0.78, xmin=0.01, xmax=0.99, color="#333", lw=0.6)

for i, (sys_, met_, tau_, pv_) in enumerate(rows_data):
    y    = 0.72 - i * 0.10
    col  = C_EEG if sys_ == "EEG" else C_MKT
    badge, bcol = tau_badge(tau_)
    axV.text(XS[0], y, sys_,  ha="left", color=col,    fontsize=8, fontfamily="monospace")
    axV.text(XS[1], y, met_,  ha="left", color="white", fontsize=8, fontfamily="monospace")
    axV.text(XS[2], y, f"{tau_:+.3f}", ha="left", color="white", fontsize=8, fontfamily="monospace")
    pv_col = C_GOOD if pv_ < 0.05 else "#777"
    axV.text(XS[3], y, f"{pv_:.4f}", ha="left", color=pv_col, fontsize=8, fontfamily="monospace")
    axV.text(XS[4], y, badge, ha="left", color=bcol,   fontsize=8, fontfamily="monospace")

pos = sum(1 for *_, tau_, _ in rows_data if tau_ > 0.1)
pct = pos / len(rows_data)
if   pct >= 0.67: verdict, vc = "HYPOTHESIS SUPPORTED",           C_GOOD
elif pct >= 0.34: verdict, vc = "HYPOTHESIS PARTIALLY SUPPORTED", C_WARN
else:             verdict, vc = "HYPOTHESIS WEAKLY SUPPORTED",    C_BAD

detail_map = {
    C_GOOD: "Both EEG and market show consistent CSD signals (rising variance, autocorr, skewness) before collapse.",
    C_WARN: "Some EWS rise consistently; others do not - mixed evidence for a universal failure signature.",
    C_BAD:  "EWS trends not consistently rising; CSD signal is weak in this single-event sample.",
}
detail = detail_map[vc]

axV.text(0.5, 0.06, f"{verdict}   ({pos}/{len(rows_data)} metrics positive)",
         ha="center", va="center", fontsize=12, color=vc,
         fontfamily="monospace", fontweight="bold")
axV.text(0.5, 0.01, detail,
         ha="center", va="center", fontsize=8.5, color="#aaa", fontfamily="monospace")

plt.savefig("critical_transitions_results.png", dpi=140, bbox_inches="tight", facecolor=BG)
plt.show()
print("Figure saved as critical_transitions_results.png")

## Conclusion, Assumptions and Caveats

In [ ]:
summary = """
HYPOTHESIS
  EEG (pre-seizure) and S&P 500 (pre-crash) both show rising variance,
  autocorrelation, and |skewness| consistent with Critical Slowing Down.

ASSUMPTIONS
  1. Local stationarity within each rolling window.
  2. Single event each (one seizure, one crash) - generalisation is limited.
  3. Linear detrend (scipy.signal.detrend) sufficient to remove slow trends.
  4. Min-max normalisation allows cross-system visual comparison.
  5. Window sizes: EEG 500 pts (~20 s effective), Market 30 trading days.
  6. EEG channel FP1-F7 (frontal) is representative of seizure activity.

CAVEATS
  1. False positives - rising EWS does not guarantee an upcoming event.
  2. Hindsight bias - crash/seizure timing was known in advance here.
  3. Market externalities (policy, geopolitics) not captured by CSD models.
  4. Real EEG carries eye/muscle artifacts; ICA artifact removal not applied.
  5. N=1 problem - statistical reliability requires many events.
  6. Single-channel EEG misses network-level dynamics.

NEXT STEPS
  - Test across 20+ seizures and 5+ market crashes.
  - Add spectral EWS: Hurst exponent, 1/f (power-law) exponent.
  - Apply graph-based EWS on full 23-channel EEG network.
  - Extend to climate, ecology, social-network tipping points.

REFERENCES
  Scheffer et al. (2009)  Nature 461:53-59   - CSD theory
  Dakos et al. (2008)     PNAS               - Kendall tau EWS method
  Meisel & Kuehn (2012)   PLOS ONE           - CSD before epileptic seizures
  Goldberger et al. (2000) Circulation       - PhysioNet / CHB-MIT database
"""
print(summary)